# Spaceship Titanic — アンサンブル（RF + LightGBM）

01・02 で **同じ特徴量** のまま RF / LightGBM を試した結果:

| | CV | LB |
|---|---|---|
| 01 RF | ~0.8035 | 0.7978 |
| 02 LightGBM | ~0.8098 | 0.8045 |

**考え:** LightGBM の方が良いが、RF と LGBM は学習の仕方が違うので、**外し方も違う**可能性がある。

**判断:** 特徴量は変えず、**soft voting**（各モデルの「転送確率」の平均）で組み合わせる。

| | 01 | 02 | 03 |
|---|---|---|---|
| 特徴量・前処理 | baseline | 同じ | **同じ** |
| モデル | RF のみ | LGBM のみ | **RF + LGBM soft voting** |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

## 1. データ読み込み・特徴量

`01_baseline.ipynb` / `02_lightgbm.ipynb` と **同じ FE** です。

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

SPEND_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def add_group_size(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    combined = pd.concat(
        [
            train_df.assign(_split="train"),
            test_df.assign(_split="test"),
        ],
        ignore_index=True,
    )
    combined["GroupId"] = combined["PassengerId"].str.split("_", n=1).str[0]
    combined["GroupSize"] = combined.groupby("GroupId")["GroupId"].transform("count")

    train_out = combined.loc[combined["_split"] == "train"].drop(columns=["_split"])
    test_out = combined.loc[combined["_split"] == "test"].drop(columns=["_split"])
    return train_out, test_out


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    cabin_parts = out["Cabin"].astype(str).str.split("/", expand=True)
    out["Deck"] = cabin_parts[0].replace("nan", np.nan)
    out["CabinNum"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    out["Side"] = cabin_parts[2].replace("nan", np.nan)

    out["TotalSpend"] = out[SPEND_COLS].fillna(0).sum(axis=1)
    out["IsAlone"] = (out["GroupSize"] == 1).astype(int)

    return out


train_gs, test_gs = add_group_size(train, test)
train_fe = add_features(train_gs)
test_fe = add_features(test_gs)

FEATURE_COLUMNS = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "Age",
    "VIP",
    *SPEND_COLS,
    "TotalSpend",
    "Deck",
    "CabinNum",
    "Side",
    "GroupSize",
    "IsAlone",
]

X = train_fe[FEATURE_COLUMNS]
y = train_fe["Transported"].astype(int)
X_test = test_fe[FEATURE_COLUMNS]

numeric_features = [
    "Age",
    *SPEND_COLS,
    "TotalSpend",
    "CabinNum",
    "GroupSize",
    "IsAlone",
]
categorical_features = ["HomePlanet", "CryoSleep", "Destination", "VIP", "Deck", "Side"]

print(f"train: {train.shape}, features: {len(FEATURE_COLUMNS)}")

## 2. アンサンブルパイプライン

**soft voting** … 各モデルが出す「Transported=True の確率」を平均し、0.5 より大きければ True と予測します。

ハイパーパラメータは 01（RF）・02（LGBM）と **同じ** に揃え、公平に比較します。

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    random_state=RANDOM_STATE,
    verbose=-1,
    n_jobs=-1,
)

ensemble = VotingClassifier(
    estimators=[("rf", rf), ("lgbm", lgbm)],
    voting="soft",
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ensemble),
])

## 3. 単体 vs アンサンブルの CV 比較

RF 単体・LGBM 単体・アンサンブルの 3 つを並べて見ます。

- アンサンブルが **両単体より上** → 相補できている
- **LGBM 単体より下** → この組み合わせは LB 提出の優先度低め

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in [
    ("RF", rf),
    ("LightGBM", lgbm),
    ("RF+LGBM soft", ensemble),
]:
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

print("\n参考 LB: RF 0.7978, LightGBM 0.8045")

## 4. 提出ファイル作成

`output/submission_ensemble.csv` に保存します。

In [ ]:
pipeline.fit(X, y)
test_pred = pipeline.predict(X_test)

submission = sample_submission.copy()
submission["Transported"] = test_pred.astype(bool)

submission_path = OUTPUT_DIR / "submission_ensemble.csv"
submission.to_csv(submission_path, index=False)

print(f"saved: {submission_path.resolve()}")
submission.head(10)

In [ ]:
# Kaggle へ提出（任意）
# !uv run kaggle competitions submit -c spaceship-titanic -f ../output/submission_ensemble.csv -m "ensemble rf+lgbm soft voting"